# Single-Node Inference

Establish a controlled denominator for all later distributed comparisons.

## Objectives

- Launch or connect to one server and verify model identity and readiness.
- Measure startup separately from steady-state latency and throughput.
- Sweep controlled prompt and output lengths while recording memory and correctness.
- Retain raw per-request results, including failures, before aggregation.

## Background

A single-node baseline defines the denominator needed to interpret distributed results. Startup, warm-up, and steady state require separate measurement boundaries.

## Prediction

For one request at a time on a single DGX Spark, inference behavior should differ between prompt processing and autoregressive generation.

Specifically:

- increasing prompt length while keeping generated length fixed should primarily increase time to first token;
- increasing generated length while keeping prompt length fixed should primarily increase total latency;
- output-token throughput should be relatively stable across sufficiently long generations, but short generations should show lower apparent throughput because fixed request and scheduling overheads make up a larger fraction of total latency;
- the first successful request after server readiness should be slower than later steady-state requests because runtime initialization, kernel loading, graph construction, cache population, or similar one-time work may still occur;
- repeated deterministic requests with the same prompt and sampling configuration should return the same token sequence unless the serving stack introduces nondeterminism;
- no request in the initial single-request benchmark should fail or exceed the configured timeout.

These predictions are falsified if prompt length has no measurable relationship with time to first token, generated length has no measurable relationship with total latency, warm-up requests are not distinguishable from measured requests, or nominally deterministic repeated requests return inconsistent token sequences.

## Environment

In [42]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/01-distributed-inference


## Experiment

Complete configuration placeholders before running active measurement cells.

### Benchmark configuration and deterministic prompts

In [43]:
from dataclasses import asdict, dataclass
from itertools import product

import pandas as pd


MODEL_ID = "google/gemma-4-E4B-it"
MODEL_REVISION = "ee0ef6023621cff504d758262d4e04895a5af4a2"

HOST = "127.0.0.1"
PORT = 8000
ENDPOINT = f"http://{HOST}:{PORT}"
MODEL = MODEL_ID

# Start with an externally managed server. This keeps server launch and
# inference measurement separate during the first experiment.
SERVER_MODE = "notebook_subprocess"

# Initial factorial workload:
#
# - 32 tokens represents a small interactive prompt.
# - 512 tokens exposes a meaningful prefill difference without making the
#   first experiment unnecessarily expensive.
# - 16 generated tokens emphasizes fixed request overhead.
# - 128 generated tokens provides a more useful decode-throughput interval.
PROMPT_TOKEN_COUNTS = (32, 512)
GENERATED_TOKEN_COUNTS = (16, 128)

WARMUP_COUNT = 3
REPETITIONS = 5
REQUEST_TIMEOUT_S = 300.0

SAMPLING = {
    "temperature": 0.0,
    "top_p": 1.0,
    "seed": 20260806,
}


@dataclass(frozen=True, slots=True)
class BenchmarkConfiguration:
    endpoint: str
    model: str
    model_revision: str
    server_mode: str
    prompt_token_counts: tuple[int, ...]
    generated_token_counts: tuple[int, ...]
    warmup_count: int
    repetitions: int
    request_timeout_s: float
    temperature: float
    top_p: float
    seed: int


benchmark_configuration = BenchmarkConfiguration(
    endpoint=ENDPOINT,
    model=MODEL,
    model_revision=MODEL_REVISION,
    server_mode=SERVER_MODE,
    prompt_token_counts=PROMPT_TOKEN_COUNTS,
    generated_token_counts=GENERATED_TOKEN_COUNTS,
    warmup_count=WARMUP_COUNT,
    repetitions=REPETITIONS,
    request_timeout_s=REQUEST_TIMEOUT_S,
    temperature=SAMPLING["temperature"],
    top_p=SAMPLING["top_p"],
    seed=SAMPLING["seed"],
)

pd.Series(asdict(benchmark_configuration), name="value")

endpoint                                     http://127.0.0.1:8000
model                                        google/gemma-4-E4B-it
model_revision            ee0ef6023621cff504d758262d4e04895a5af4a2
server_mode                                    notebook_subprocess
prompt_token_counts                                      (32, 512)
generated_token_counts                                   (16, 128)
warmup_count                                                     3
repetitions                                                      5
request_timeout_s                                            300.0
temperature                                                    0.0
top_p                                                          1.0
seed                                                      20260806
Name: value, dtype: object

### Measurement contract

This notebook uses one request at a time. It does not measure batching, concurrency, queueing, or distributed execution.

For each request:

- **Prompt tokens** are the input token count reported by the server, checked against the intended workload size.
- **Output tokens** are the generated token count reported by the server.
- **Time to first token (TTFT)** is the elapsed monotonic wall-clock time from immediately before request submission until the first non-empty streamed token or text fragment is received.
- **End-to-end latency** is the elapsed monotonic wall-clock time from immediately before request submission until the complete response stream has been consumed.
- **Generation interval** is `end-to-end latency - TTFT`.
- **Output-token throughput** is the number of returned output tokens divided by the generation interval.
- **End-to-end token throughput** is the number of returned output tokens divided by end-to-end latency.

Warm-up requests are retained separately and are never included in steady-state aggregates.

All raw request records, including failed requests and token-count mismatches, are retained before aggregation.

In [44]:
def validate_benchmark_configuration(
    configuration: BenchmarkConfiguration,
) -> None:
    if not configuration.endpoint.startswith(("http://", "https://")):
        raise ValueError("endpoint must be an HTTP or HTTPS URL")

    if not configuration.model:
        raise ValueError("model must not be empty")

    if configuration.server_mode not in {"external", "notebook_subprocess"}:
        raise ValueError("server_mode must be 'external' or 'notebook_subprocess'")

    for name, values in (
        ("prompt_token_counts", configuration.prompt_token_counts),
        ("generated_token_counts", configuration.generated_token_counts),
    ):
        if not values:
            raise ValueError(f"{name} must not be empty")
        if any(
            isinstance(value, bool) or not isinstance(value, int) or value <= 0
            for value in values
        ):
            raise ValueError(f"{name} must contain positive integers")
        if len(values) != len(set(values)):
            raise ValueError(f"{name} must not contain duplicates")

    if configuration.warmup_count < 0:
        raise ValueError("warmup_count must be non-negative")

    if configuration.repetitions <= 0:
        raise ValueError("repetitions must be positive")

    if configuration.request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be positive")

    if configuration.temperature < 0:
        raise ValueError("temperature must be non-negative")

    if not 0 < configuration.top_p <= 1:
        raise ValueError("top_p must be in the interval (0, 1]")


validate_benchmark_configuration(benchmark_configuration)

workload_matrix = pd.DataFrame(
    [
        {
            "prompt_tokens_target": prompt_tokens,
            "requested_output_tokens": generated_tokens,
        }
        for prompt_tokens, generated_tokens in product(
            PROMPT_TOKEN_COUNTS,
            GENERATED_TOKEN_COUNTS,
        )
    ]
)

expected_measured_requests = len(workload_matrix) * REPETITIONS

print(f"Warm-up requests: {WARMUP_COUNT}")
print(f"Workload configurations: {len(workload_matrix)}")
print(f"Repetitions per configuration: {REPETITIONS}")
print(f"Expected measured requests: {expected_measured_requests}")

workload_matrix

Warm-up requests: 3
Workload configurations: 4
Repetitions per configuration: 5
Expected measured requests: 20


,prompt_tokens_target,requested_output_tokens
0,32,16
1,32,128
2,512,16
3,512,128


### Exact-length prompt construction

Prompt lengths must be established with the tokenizer belonging to the pinned model revision.

The notebook environment does not assume that `transformers` is installed. Tokenization therefore runs in the previously verified vLLM container with the host Hugging Face cache mounted read-only.

The generated prompts use repeated neutral prose. Their purpose is to produce controlled token counts, not to evaluate factual knowledge or instruction-following quality.

The target count includes any tokens added by the model's chat template because those tokens are part of the server's actual prefill input.

In [45]:
import json
import shlex
import subprocess
from pathlib import Path


def read_env_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}

    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        key, separator, value = line.partition("=")
        if not separator:
            raise ValueError(f"Invalid configuration line: {raw_line!r}")

        values[key.strip()] = value.strip()

    return values


cluster = read_env_file(repository_root / "config" / "cluster.env")

VLLM_IMAGE = cluster["VLLM_IMAGE"]
HF_CACHE_ROOT = Path.home() / ".cache" / "huggingface"
MODEL_CACHE_DIRECTORY = "models--google--gemma-4-E4B-it"
MODEL_SNAPSHOT_PATH = (
    HF_CACHE_ROOT / "hub" / MODEL_CACHE_DIRECTORY / "snapshots" / MODEL_REVISION
)

if not MODEL_SNAPSHOT_PATH.is_dir():
    raise FileNotFoundError(
        f"Pinned model snapshot is not available: {MODEL_SNAPSHOT_PATH}"
    )

tokenizer_environment = pd.Series(
    {
        "vllm_image": VLLM_IMAGE,
        "host_cache_root": str(HF_CACHE_ROOT),
        "model_snapshot_path": str(MODEL_SNAPSHOT_PATH),
        "model_revision": MODEL_REVISION,
    },
    name="value",
)

tokenizer_environment

vllm_image                                                     vllm-node
host_cache_root                           /home/coert/.cache/huggingface
model_snapshot_path    /home/coert/.cache/huggingface/hub/models--goo...
model_revision                  ee0ef6023621cff504d758262d4e04895a5af4a2
Name: value, dtype: str

In [46]:
import json
import subprocess
import tempfile


TOKENIZER_SCRIPT = r"""
import json
import sys

from transformers import AutoTokenizer


configuration_path = sys.argv[1]
with open(configuration_path, encoding="utf-8") as file:
    configuration = json.load(file)

tokenizer = AutoTokenizer.from_pretrained(
    configuration["model_path"],
    local_files_only=True,
    trust_remote_code=False,
)

seed_text = (
    "The system processes a fixed sequence of neutral words for a controlled "
    "inference benchmark. Each sentence contains ordinary language and adds "
    "no request for external facts. "
)

instruction = (
    "Continue the following text with a concise neutral sentence. "
    "Do not use headings or lists."
)


def serialize_messages(content: str) -> str:
    messages = [
        {
            "role": "user",
            "content": f"{instruction}\n\n{content}",
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def encode(text: str) -> list[int]:
    return tokenizer.encode(text, add_special_tokens=False)


def make_exact_prompt(target_tokens: int) -> dict:
    content = ""

    while len(encode(serialize_messages(content))) < target_tokens:
        content += seed_text

    serialized = serialize_messages(content)
    serialized_ids = encode(serialized)
    exact_ids = serialized_ids[:target_tokens]

    prompt = tokenizer.decode(
        exact_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )
    verified_ids = encode(prompt)

    if verified_ids != exact_ids:
        raise RuntimeError(
            f"Target {target_tokens}: decoded prompt does not reproduce "
            "the selected token sequence"
        )

    return {
        "target_tokens": target_tokens,
        "actual_tokens": len(verified_ids),
        "prompt": prompt,
        "token_ids": verified_ids,
    }


result = {
    "tokenizer_name_or_path": tokenizer.name_or_path,
    "tokenizer_class": type(tokenizer).__name__,
    "bos_token_id": tokenizer.bos_token_id,
    "eos_token_id": tokenizer.eos_token_id,
    "prompts": [
        make_exact_prompt(target)
        for target in configuration["targets"]
    ],
}

json.dump(result, sys.stdout)
"""


def construct_exact_prompts(
    *,
    image: str,
    cache_root: Path,
    model_snapshot_path: Path,
    targets: tuple[int, ...],
) -> dict:
    configuration = {
        "model_path": str(model_snapshot_path),
        "targets": list(targets),
    }

    with tempfile.TemporaryDirectory(prefix="dgx-prompt-") as temporary_directory:
        temporary_path = Path(temporary_directory)
        script_path = temporary_path / "construct_prompts.py"
        configuration_path = temporary_path / "configuration.json"

        script_path.write_text(TOKENIZER_SCRIPT)
        configuration_path.write_text(json.dumps(configuration))

        container_directory = "/tmp/dgx-prompt"

        command = [
            "docker",
            "run",
            "--rm",
            "--network",
            "none",
            "--entrypoint",
            "/bin/sh",
            "-v",
            f"{cache_root}:{cache_root}:ro",
            "-v",
            f"{temporary_path}:{container_directory}:ro",
            image,
            "-lc",
            (
                "python3 "
                f"{container_directory}/construct_prompts.py "
                f"{container_directory}/configuration.json"
            ),
        ]

        completed = subprocess.run(
            command,
            text=True,
            capture_output=True,
            timeout=REQUEST_TIMEOUT_S,
            check=False,
        )

    if completed.returncode != 0:
        raise RuntimeError(
            "Tokenizer container failed.\n\n"
            f"stdout:\n{completed.stdout}\n\n"
            f"stderr:\n{completed.stderr}"
        )

    return json.loads(completed.stdout)


prompt_construction = construct_exact_prompts(
    image=VLLM_IMAGE,
    cache_root=HF_CACHE_ROOT,
    model_snapshot_path=MODEL_SNAPSHOT_PATH,
    targets=PROMPT_TOKEN_COUNTS,
)

prompt_construction_summary = pd.DataFrame(
    [
        {
            "prompt_id": f"prompt_{item['target_tokens']}",
            "target_tokens": item["target_tokens"],
            "actual_tokens": item["actual_tokens"],
            "character_count": len(item["prompt"]),
            "token_id_checksum": sum(item["token_ids"]),
        }
        for item in prompt_construction["prompts"]
    ]
)

prompt_construction_summary

,prompt_id,target_tokens,actual_tokens,character_count,token_id_checksum
0,prompt_32,32,32,167,738859
1,prompt_512,512,512,3158,12317405


In [47]:
PROMPTS = {
    f"prompt_{item['target_tokens']}": item["prompt"]
    for item in prompt_construction["prompts"]
}

PROMPT_TOKEN_IDS = {
    f"prompt_{item['target_tokens']}": tuple(item["token_ids"])
    for item in prompt_construction["prompts"]
}

expected_prompt_ids = {f"prompt_{target}" for target in PROMPT_TOKEN_COUNTS}

if set(PROMPTS) != expected_prompt_ids:
    raise AssertionError(f"Unexpected prompt IDs: {sorted(PROMPTS)}")

for target in PROMPT_TOKEN_COUNTS:
    prompt_id = f"prompt_{target}"
    actual = len(PROMPT_TOKEN_IDS[prompt_id])

    if actual != target:
        raise AssertionError(f"{prompt_id}: expected {target} tokens, found {actual}")

print(f"Tokenizer class: {prompt_construction['tokenizer_class']}")
print(f"Tokenizer snapshot: {prompt_construction['tokenizer_name_or_path']}")
print(f"Registered prompts: {sorted(PROMPTS)}")

prompt_construction_summary

Tokenizer class: GemmaTokenizer
Tokenizer snapshot: /home/coert/.cache/huggingface/hub/models--google--gemma-4-E4B-it/snapshots/ee0ef6023621cff504d758262d4e04895a5af4a2
Registered prompts: ['prompt_32', 'prompt_512']


,prompt_id,target_tokens,actual_tokens,character_count,token_id_checksum
0,prompt_32,32,32,167,738859
1,prompt_512,512,512,3158,12317405


### Server lifecycle

For notebook-managed mode, construct an argument list only after selecting a model. Retain the exact `Popen` handle and terminate only that process during cleanup.

In [48]:
def inspect_image_configuration(
    image: str,
) -> dict[str, object]:
    completed = subprocess.run(
        [
            "docker",
            "image",
            "inspect",
            image,
            "--format",
            "{{json .Config}}",
        ],
        text=True,
        capture_output=True,
        timeout=30.0,
        check=False,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            "Could not inspect Docker image configuration.\n\n"
            f"stdout:\n{completed.stdout}\n\n"
            f"stderr:\n{completed.stderr}"
        )

    configuration = json.loads(completed.stdout)

    return {
        "image": image,
        "entrypoint": configuration.get("Entrypoint"),
        "cmd": configuration.get("Cmd"),
        "working_directory": configuration.get("WorkingDir"),
    }


image_configuration = inspect_image_configuration(VLLM_IMAGE)

pd.Series(image_configuration, name="value")

image                                         vllm-node
entrypoint           [/opt/nvidia/nvidia_entrypoint.sh]
cmd                                                None
working_directory                       /workspace/vllm
Name: value, dtype: object

In [49]:
import shlex
import subprocess
import uuid


SERVER_CONTAINER_NAME = f"dgx-spark-single-node-{uuid.uuid4().hex[:8]}"

SERVER_LOG_PATH = (
    repository_root
    / "experiments"
    / "01-distributed-inference"
    / f"{SERVER_CONTAINER_NAME}.log"
)

# Conservative initial envelope for the single-request baseline.
#
# DGX Spark exposes unified CPU/GPU memory, so leave substantial host headroom.
VLLM_GPU_MEMORY_UTILIZATION = 0.20

# The measured workload needs at most 512 prompt + 128 generated tokens.
# 1024 preserves debugging headroom without exposing the model's very large
# tokenizer/model limit to this first experiment.
VLLM_MAX_MODEL_LEN = 1_024

server_process: subprocess.Popen[str] | None = None
server_log_file = None
server_started_monotonic_s: float | None = None


def build_server_arguments(
    *,
    image: str,
    container_name: str,
    model_snapshot_path: Path,
    served_model_name: str,
    port: int,
    gpu_memory_utilization: float,
    max_model_len: int,
) -> list[str]:
    if not image:
        raise ValueError("image must not be empty")
    if not container_name:
        raise ValueError("container_name must not be empty")
    if not model_snapshot_path.is_dir():
        raise FileNotFoundError(model_snapshot_path)
    if not served_model_name:
        raise ValueError("served_model_name must not be empty")
    if not 1 <= port <= 65535:
        raise ValueError("port must be between 1 and 65535")
    if not 0 < gpu_memory_utilization <= 1:
        raise ValueError("gpu_memory_utilization must be in (0, 1]")

    required_model_len = max(PROMPT_TOKEN_COUNTS) + max(GENERATED_TOKEN_COUNTS)
    if max_model_len < required_model_len:
        raise ValueError(
            f"max_model_len={max_model_len} is smaller than the "
            f"largest benchmark sequence, {required_model_len}"
        )

    cache_root = Path.home() / ".cache" / "huggingface"

    return [
        "docker",
        "run",
        "--rm",
        "--name",
        container_name,
        "--gpus",
        "all",
        "--ipc",
        "host",
        "--network",
        "host",
        "-v",
        f"{cache_root}:{cache_root}:ro",
        image,
        "vllm",
        "serve",
        str(model_snapshot_path),
        "--served-model-name",
        served_model_name,
        "--host",
        HOST,
        "--port",
        str(port),
        "--gpu-memory-utilization",
        str(gpu_memory_utilization),
        "--max-model-len",
        str(max_model_len),
    ]


SERVER_ARGUMENTS = build_server_arguments(
    image=VLLM_IMAGE,
    container_name=SERVER_CONTAINER_NAME,
    model_snapshot_path=MODEL_SNAPSHOT_PATH,
    served_model_name=MODEL,
    port=PORT,
    gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    max_model_len=VLLM_MAX_MODEL_LEN,
)

pd.Series(
    {
        "container_name": SERVER_CONTAINER_NAME,
        "log_path": str(SERVER_LOG_PATH),
        "gpu_memory_utilization": VLLM_GPU_MEMORY_UTILIZATION,
        "max_model_len": VLLM_MAX_MODEL_LEN,
        "maximum_benchmark_sequence_tokens": (
            max(PROMPT_TOKEN_COUNTS) + max(GENERATED_TOKEN_COUNTS)
        ),
        "command": shlex.join(SERVER_ARGUMENTS),
    },
    name="value",
)

container_name                                          dgx-spark-single-node-68ca5e09
log_path                             /home/coert/workspace/dgx-spark-lab/experiment...
gpu_memory_utilization                                                             0.2
max_model_len                                                                     1024
maximum_benchmark_sequence_tokens                                                  640
command                              docker run --rm --name dgx-spark-single-node-6...
Name: value, dtype: object

In [50]:
def inspect_actual_vllm_serve_help(
    image: str,
) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        [
            "docker",
            "run",
            "--rm",
            "--gpus",
            "all",
            image,
            "vllm",
            "serve",
            "--help",
        ],
        text=True,
        capture_output=True,
        timeout=60.0,
        check=False,
    )


actual_serve_help = inspect_actual_vllm_serve_help(VLLM_IMAGE)

pd.Series(
    {
        "returncode": actual_serve_help.returncode,
        "stdout_characters": len(actual_serve_help.stdout),
        "stderr_characters": len(actual_serve_help.stderr),
        "contains_served_model_name": (
            "--served-model-name"
            in (actual_serve_help.stdout + actual_serve_help.stderr)
        ),
        "contains_host": (
            "--host" in (actual_serve_help.stdout + actual_serve_help.stderr)
        ),
        "contains_port": (
            "--port" in (actual_serve_help.stdout + actual_serve_help.stderr)
        ),
    },
    name="value",
)

returncode                        0
stdout_characters              5308
stderr_characters                 0
contains_served_model_name    False
contains_host                 False
contains_port                 False
Name: value, dtype: object

In [51]:
if actual_serve_help.returncode != 0:
    print("stdout:")
    print(actual_serve_help.stdout[-8_000:])
    print("\nstderr:")
    print(actual_serve_help.stderr[-8_000:])

    raise RuntimeError("`vllm serve --help` failed inside the pinned image")

print("The actual `vllm serve` command is available.")

The actual `vllm serve` command is available.


In [52]:
def inspect_named_container(
    container_name: str,
) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        [
            "docker",
            "inspect",
            container_name,
            "--format",
            "{{json .State}}",
        ],
        text=True,
        capture_output=True,
        timeout=10.0,
        check=False,
    )


existing_container = inspect_named_container(SERVER_CONTAINER_NAME)

if existing_container.returncode == 0:
    raise RuntimeError(
        f"Container {SERVER_CONTAINER_NAME!r} already exists; "
        "refusing to replace or stop it implicitly"
    )

if SERVER_LOG_PATH.exists():
    raise FileExistsError(f"Refusing to overwrite existing log: {SERVER_LOG_PATH}")

print(f"Container name is unused: {SERVER_CONTAINER_NAME}")
print(f"Log path is unused: {SERVER_LOG_PATH}")

Container name is unused: dgx-spark-single-node-68ca5e09
Log path is unused: /home/coert/workspace/dgx-spark-lab/experiments/01-distributed-inference/dgx-spark-single-node-68ca5e09.log


In [53]:
import time

if SERVER_MODE != "notebook_subprocess":
    raise RuntimeError("This cell requires SERVER_MODE='notebook_subprocess'")

if server_process is not None:
    raise RuntimeError("This notebook kernel already holds a server process handle")

server_log_file = SERVER_LOG_PATH.open(
    "w",
    encoding="utf-8",
    buffering=1,
)

server_started_monotonic_s = time.monotonic()

try:
    server_process = subprocess.Popen(
        SERVER_ARGUMENTS,
        stdin=subprocess.DEVNULL,
        stdout=server_log_file,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
except Exception:
    server_log_file.close()
    server_log_file = None
    server_started_monotonic_s = None
    raise

print(f"Started Docker client process PID: {server_process.pid}")
print(f"Container name: {SERVER_CONTAINER_NAME}")
print(f"Server log: {SERVER_LOG_PATH}")

Started Docker client process PID: 3924541
Container name: dgx-spark-single-node-68ca5e09
Server log: /home/coert/workspace/dgx-spark-lab/experiments/01-distributed-inference/dgx-spark-single-node-68ca5e09.log


In [54]:
INITIAL_LAUNCH_OBSERVATION_S = 5.0

time.sleep(INITIAL_LAUNCH_OBSERVATION_S)

server_returncode = server_process.poll()

server_log_file.flush()
server_log_text = SERVER_LOG_PATH.read_text(
    encoding="utf-8",
    errors="replace",
)

launch_summary = pd.Series(
    {
        "docker_client_pid": server_process.pid,
        "docker_client_returncode": server_returncode,
        "observation_interval_s": INITIAL_LAUNCH_OBSERVATION_S,
        "log_characters": len(server_log_text),
        "container_name": SERVER_CONTAINER_NAME,
        "server_endpoint": ENDPOINT,
    },
    name="value",
)

launch_summary

docker_client_pid                                  3924541
docker_client_returncode                              None
observation_interval_s                                 5.0
log_characters                                         523
container_name              dgx-spark-single-node-68ca5e09
server_endpoint                      http://127.0.0.1:8000
Name: value, dtype: object

In [55]:
LOG_TAIL_CHARACTERS = 8_000

print(server_log_text[-LOG_TAIL_CHARACTERS:])

if server_returncode is not None:
    raise RuntimeError(
        "The vLLM container exited during initial launch. "
        f"Docker client return code: {server_returncode}. "
        "Inspect the saved log above."
    )

print(
    "The container remained active through the initial "
    f"{INITIAL_LAUNCH_OBSERVATION_S:.1f} s observation interval."
)


== CUDA ==

CUDA Version 13.0.2

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.


The container remained active through the initial 5.0 s observation interval.


### Readiness and request scaffolds

Poll the notebook-managed server until its OpenAI-compatible model endpoint is ready, while also detecting an early server-process exit.

In [56]:
import json
import time
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


def request_json(
    url: str,
    *,
    timeout_s: float,
) -> tuple[int, dict[str, Any], dict[str, str]]:
    request = Request(
        url,
        headers={
            "Accept": "application/json",
            "User-Agent": "dgx-spark-lab/01-single-node-inference",
        },
        method="GET",
    )

    with urlopen(request, timeout=timeout_s) as response:
        status_code = response.status
        headers = {key.lower(): value for key, value in response.headers.items()}
        body = response.read().decode("utf-8")

    parsed = json.loads(body)
    if not isinstance(parsed, dict):
        raise TypeError(
            f"Expected a JSON object from {url}, received {type(parsed).__name__}"
        )

    return status_code, parsed, headers


def wait_until_ready(
    endpoint: str,
    timeout_s: float,
    *,
    poll_interval_s: float = 1.0,
    request_timeout_s: float = 5.0,
    process: subprocess.Popen[str] | None = None,
) -> dict[str, Any]:
    """Poll the OpenAI-compatible model-list endpoint until it succeeds."""

    if timeout_s <= 0:
        raise ValueError("timeout_s must be positive")
    if poll_interval_s <= 0:
        raise ValueError("poll_interval_s must be positive")
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be positive")

    models_url = f"{endpoint.rstrip('/')}/v1/models"
    started_s = time.monotonic()
    deadline_s = started_s + timeout_s

    attempts: list[dict[str, Any]] = []

    while True:
        if process is not None and process.poll() is not None:
            return {
                "ready": False,
                "models_url": models_url,
                "elapsed_s": time.monotonic() - started_s,
                "attempt_count": len(attempts),
                "advertised_models": (),
                "response_payload": None,
                "response_headers": {},
                "attempts": attempts,
            }

        attempt_started_s = time.monotonic()

        try:
            status_code, payload, headers = request_json(
                models_url,
                timeout_s=request_timeout_s,
            )

            attempts.append(
                {
                    "attempt": len(attempts) + 1,
                    "elapsed_s": time.monotonic() - started_s,
                    "request_latency_s": (time.monotonic() - attempt_started_s),
                    "status_code": status_code,
                    "error_type": None,
                    "error": None,
                }
            )

            model_records = payload.get("data")
            if not isinstance(model_records, list):
                raise ValueError("The /v1/models response does not contain a data list")

            advertised_models = tuple(
                record.get("id")
                for record in model_records
                if isinstance(record, dict) and isinstance(record.get("id"), str)
            )

            return {
                "ready": True,
                "models_url": models_url,
                "elapsed_s": time.monotonic() - started_s,
                "attempt_count": len(attempts),
                "advertised_models": advertised_models,
                "response_payload": payload,
                "response_headers": headers,
                "attempts": attempts,
            }

        except HTTPError as error:
            error_body = error.read().decode(
                "utf-8",
                errors="replace",
            )
            error_type = type(error).__name__
            error_message = (
                f"HTTP {error.code}: {error.reason}; body={error_body[:500]!r}"
            )

        except (
            URLError,
            TimeoutError,
            ConnectionError,
            json.JSONDecodeError,
            OSError,
            TypeError,
            ValueError,
        ) as error:
            error_type = type(error).__name__
            error_message = str(error)

        attempts.append(
            {
                "attempt": len(attempts) + 1,
                "elapsed_s": time.monotonic() - started_s,
                "request_latency_s": (time.monotonic() - attempt_started_s),
                "status_code": None,
                "error_type": error_type,
                "error": error_message,
            }
        )

        now_s = time.monotonic()
        if now_s >= deadline_s:
            return {
                "ready": False,
                "models_url": models_url,
                "elapsed_s": now_s - started_s,
                "attempt_count": len(attempts),
                "advertised_models": (),
                "response_payload": None,
                "response_headers": {},
                "attempts": attempts,
            }

        time.sleep(
            min(
                poll_interval_s,
                max(0.0, deadline_s - now_s),
            )
        )

In [57]:
SERVER_STARTUP_TIMEOUT_S = 900.0

readiness_result = wait_until_ready(
    ENDPOINT,
    timeout_s=SERVER_STARTUP_TIMEOUT_S,
    poll_interval_s=2.0,
    request_timeout_s=5.0,
    process=server_process,
)

server_ready_monotonic_s = time.monotonic()

startup_duration_s = (
    server_ready_monotonic_s - server_started_monotonic_s
    if readiness_result["ready"]
    else None
)

readiness_summary = pd.Series(
    {
        "ready": readiness_result["ready"],
        "models_url": readiness_result["models_url"],
        "startup_duration_s": startup_duration_s,
        "readiness_attempt_count": readiness_result["attempt_count"],
        "advertised_models": readiness_result["advertised_models"],
        "configured_model": MODEL,
        "configured_model_advertised": (MODEL in readiness_result["advertised_models"]),
        "docker_client_returncode": server_process.poll(),
    },
    name="value",
)

readiness_summary

ready                                                     True
models_url                     http://127.0.0.1:8000/v1/models
startup_duration_s                                  245.133212
readiness_attempt_count                                    121
advertised_models                     (google/gemma-4-E4B-it,)
configured_model                         google/gemma-4-E4B-it
configured_model_advertised                               True
docker_client_returncode                                  None
Name: value, dtype: object

In [58]:
readiness_attempts = pd.DataFrame(readiness_result["attempts"])

if not readiness_result["ready"]:
    server_log_file.flush()
    failed_log_text = SERVER_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )

    display(readiness_attempts.tail(10))
    print(failed_log_text[-8_000:])

    raise RuntimeError(
        f"Server did not become ready within {SERVER_STARTUP_TIMEOUT_S:.0f} seconds"
    )

if MODEL not in readiness_result["advertised_models"]:
    raise RuntimeError(
        f"Configured model {MODEL!r} was not advertised. "
        f"Advertised models: "
        f"{readiness_result['advertised_models']!r}"
    )

print(f"Server startup duration: {startup_duration_s:.3f} s")
print(f"Advertised model verified: {MODEL}")

Server startup duration: 245.133 s
Advertised model verified: google/gemma-4-E4B-it


### Completion API contract probe

Before warm-up or benchmark measurement, send one excluded diagnostic request.

This probe verifies that:

- the serialized prompt is accepted by `/v1/completions`;
- the server does not add another special token;
- server-reported prompt tokens match the tokenizer-derived target;
- `ignore_eos` produces the requested output length;
- the configured served-model identity is returned.

The probe is diagnostic only. Its latency is not part of warm-up or steady-state results.

In [59]:
from urllib.error import HTTPError
from urllib.request import Request, urlopen


def send_completion_probe(
    *,
    endpoint: str,
    model: str,
    prompt: str,
    max_tokens: int,
    timeout_s: float,
    sampling: dict[str, object],
) -> dict[str, object]:
    completions_url = f"{endpoint.rstrip('/')}/v1/completions"

    payload = {
        "model": model,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": sampling["temperature"],
        "top_p": sampling["top_p"],
        "seed": sampling["seed"],
        "stream": False,
        # The prompt already contains the tokenizer's serialized chat template
        # and special tokens.
        "add_special_tokens": False,
        # This controlled length probe must not stop early at EOS.
        "ignore_eos": True,
    }

    request = Request(
        completions_url,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Accept": "application/json",
            "Content-Type": "application/json",
            "User-Agent": "dgx-spark-lab/01-single-node-inference",
        },
        method="POST",
    )

    started_s = time.monotonic()

    try:
        with urlopen(request, timeout=timeout_s) as response:
            status_code = response.status
            response_body = response.read().decode("utf-8")
    except HTTPError as error:
        error_body = error.read().decode(
            "utf-8",
            errors="replace",
        )
        raise RuntimeError(
            f"Completion probe failed with HTTP {error.code}: "
            f"{error.reason}; body={error_body[:2_000]!r}"
        ) from error

    completed_s = time.monotonic()
    response_payload = json.loads(response_body)

    choices = response_payload.get("choices")
    if not isinstance(choices, list) or len(choices) != 1:
        raise ValueError(
            f"Expected exactly one completion choice, received {choices!r}"
        )

    choice = choices[0]
    usage = response_payload.get("usage")

    if not isinstance(choice, dict):
        raise TypeError("Completion choice is not a JSON object")
    if not isinstance(usage, dict):
        raise TypeError("Completion response does not contain usage data")

    return {
        "status_code": status_code,
        "response_id": response_payload.get("id"),
        "response_model": response_payload.get("model"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "total_tokens": usage.get("total_tokens"),
        "finish_reason": choice.get("finish_reason"),
        "latency_s": completed_s - started_s,
        "output_text": choice.get("text"),
    }


PROBE_PROMPT_ID = "prompt_32"
PROBE_OUTPUT_TOKENS = 16

completion_probe = send_completion_probe(
    endpoint=ENDPOINT,
    model=MODEL,
    prompt=PROMPTS[PROBE_PROMPT_ID],
    max_tokens=PROBE_OUTPUT_TOKENS,
    timeout_s=REQUEST_TIMEOUT_S,
    sampling=SAMPLING,
)

pd.Series(completion_probe, name="value")

status_code                                                        200
response_id                                      cmpl-9566755867373b65
response_model                                   google/gemma-4-E4B-it
prompt_tokens                                                       32
completion_tokens                                                   16
total_tokens                                                        48
finish_reason                                                   length
latency_s                                                     1.471401
output_text           each input. This sequence dictates the struct...
Name: value, dtype: object

In [60]:
expected_probe_prompt_tokens = len(PROMPT_TOKEN_IDS[PROBE_PROMPT_ID])

probe_invariants = pd.Series(
    {
        "http_status_is_200": (completion_probe["status_code"] == 200),
        "model_matches": (completion_probe["response_model"] == MODEL),
        "prompt_tokens_match": (
            completion_probe["prompt_tokens"] == expected_probe_prompt_tokens
        ),
        "completion_tokens_match": (
            completion_probe["completion_tokens"] == PROBE_OUTPUT_TOKENS
        ),
        "total_tokens_match": (
            completion_probe["total_tokens"]
            == expected_probe_prompt_tokens + PROBE_OUTPUT_TOKENS
        ),
        "finish_reason_is_length": (completion_probe["finish_reason"] == "length"),
        "output_is_nonempty": bool(completion_probe["output_text"]),
    },
    name="passed",
)

display(probe_invariants.to_frame())

if not bool(probe_invariants.all()):
    raise AssertionError(
        "The completion API contract probe failed one or more invariants"
    )

print("Completion API contract verified.")

,passed
http_status_is_200,True
model_matches,True
prompt_tokens_match,True
completion_tokens_match,True
total_tokens_match,True
finish_reason_is_length,True
output_is_nonempty,True


Completion API contract verified.


### Streaming completion contract probe

Send one additional excluded diagnostic request using the streamed completions API.

The first non-empty text fragment establishes the client-observed time to first token. The complete stream establishes end-to-end latency. This probe validates the streaming protocol and timing boundaries only; it is not included in warm-up or benchmark results.

In [ ]:
def send_streaming_completion_request(
    *,
    endpoint: str,
    model: str,
    prompt_id: str,
    prompt: str,
    requested_output_tokens: int,
    timeout_s: float,
    sampling: dict[str, object],
) -> dict[str, object]:
    completions_url = f"{endpoint.rstrip('/')}/v1/completions"
    client_request_id = f"request-{uuid.uuid4().hex}"

    payload = {
        "model": model,
        "prompt": prompt,
        "max_tokens": requested_output_tokens,
        "temperature": sampling["temperature"],
        "top_p": sampling["top_p"],
        "seed": sampling["seed"],
        "stream": True,
        "stream_options": {
            "include_usage": True,
        },
        "add_special_tokens": False,
        "ignore_eos": True,
    }

    request = Request(
        completions_url,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Accept": "text/event-stream",
            "Content-Type": "application/json",
            "User-Agent": "dgx-spark-lab/01-single-node-inference",
        },
        method="POST",
    )

    started_s = time.monotonic()
    first_token_s = None
    completed_s = None

    status_code = None
    response_id = None
    response_model = None
    finish_reason = None
    usage = None
    text_fragments: list[str] = []
    error_message = None

    try:
        with urlopen(request, timeout=timeout_s) as response:
            status_code = response.status

            for raw_line in response:
                line = raw_line.decode(
                    "utf-8",
                    errors="strict",
                ).strip()

                if not line or not line.startswith("data:"):
                    continue

                event_data = line.removeprefix("data:").strip()

                if event_data == "[DONE]":
                    break

                event = json.loads(event_data)

                if response_id is None:
                    response_id = event.get("id")
                if response_model is None:
                    response_model = event.get("model")

                event_usage = event.get("usage")
                if isinstance(event_usage, dict):
                    usage = event_usage

                choices = event.get("choices", [])
                if not isinstance(choices, list):
                    raise TypeError("Streaming event choices must be a list")

                for choice in choices:
                    if not isinstance(choice, dict):
                        raise TypeError("Streaming choice must be an object")

                    fragment = choice.get("text")
                    if isinstance(fragment, str) and fragment:
                        if first_token_s is None:
                            first_token_s = time.monotonic()

                        text_fragments.append(fragment)

                    if choice.get("finish_reason") is not None:
                        finish_reason = choice["finish_reason"]

        completed_s = time.monotonic()

        if first_token_s is None:
            raise RuntimeError("Stream completed without a non-empty text fragment")
        if not isinstance(usage, dict):
            raise RuntimeError("Stream completed without final usage data")

    except HTTPError as error:
        completed_s = time.monotonic()
        error_body = error.read().decode(
            "utf-8",
            errors="replace",
        )
        status_code = error.code
        error_message = (
            f"HTTPError: HTTP {error.code}: {error.reason}; body={error_body[:2_000]!r}"
        )

    except (
        URLError,
        TimeoutError,
        ConnectionError,
        json.JSONDecodeError,
        UnicodeDecodeError,
        OSError,
        TypeError,
        ValueError,
        RuntimeError,
    ) as error:
        completed_s = time.monotonic()
        error_message = f"{type(error).__name__}: {error}"

    output_text = "".join(text_fragments)

    ttft_s = first_token_s - started_s if first_token_s is not None else None
    latency_s = completed_s - started_s if completed_s is not None else None
    generation_interval_s = (
        completed_s - first_token_s
        if completed_s is not None and first_token_s is not None
        else None
    )

    prompt_tokens = usage.get("prompt_tokens") if isinstance(usage, dict) else None
    returned_output_tokens = (
        usage.get("completion_tokens") if isinstance(usage, dict) else None
    )

    output_tokens_per_s = (
        returned_output_tokens / generation_interval_s
        if isinstance(returned_output_tokens, int)
        and generation_interval_s is not None
        and generation_interval_s > 0
        else None
    )

    return {
        "client_request_id": client_request_id,
        "response_id": response_id,
        "response_model": response_model,
        "prompt_id": prompt_id,
        "prompt_tokens": prompt_tokens,
        "requested_output_tokens": requested_output_tokens,
        "returned_output_tokens": returned_output_tokens,
        "status_code": status_code,
        "finish_reason": finish_reason,
        "started_monotonic_s": started_s,
        "first_token_monotonic_s": first_token_s,
        "completed_monotonic_s": completed_s,
        "ttft_s": ttft_s,
        "generation_interval_s": generation_interval_s,
        "latency_s": latency_s,
        "output_tokens_per_s": output_tokens_per_s,
        "output_text": output_text,
        "status": ("ok" if error_message is None else "error"),
        "error": error_message,
    }

In [63]:
streaming_probe = send_streaming_completion_request(
    endpoint=ENDPOINT,
    model=MODEL,
    prompt_id=PROBE_PROMPT_ID,
    prompt=PROMPTS[PROBE_PROMPT_ID],
    requested_output_tokens=PROBE_OUTPUT_TOKENS,
    timeout_s=REQUEST_TIMEOUT_S,
    sampling=SAMPLING,
)

pd.Series(streaming_probe, name="value")

client_request_id                   request-c61cdedcb6a64176a51e7833df93be3b
response_id                                            cmpl-91325cf3ae921f82
response_model                                         google/gemma-4-E4B-it
prompt_id                                                          prompt_32
prompt_tokens                                                             32
requested_output_tokens                                                   16
returned_output_tokens                                                    16
status_code                                                              200
finish_reason                                                         length
started_monotonic_s                                           1427745.482181
first_token_monotonic_s                                       1427745.563521
completed_monotonic_s                                         1427746.319628
ttft_s                                                               0.08134

In [ ]:
streaming_probe_invariants = pd.Series(
    {
        "status_is_ok": (streaming_probe["status"] == "ok"),
        "http_status_is_200": (streaming_probe["status_code"] == 200),
        "model_matches": (streaming_probe["response_model"] == MODEL),
        "prompt_tokens_match": (
            streaming_probe["prompt_tokens"] == expected_probe_prompt_tokens
        ),
        "completion_tokens_match": (
            streaming_probe["returned_output_tokens"] == PROBE_OUTPUT_TOKENS
        ),
        "finish_reason_is_length": (streaming_probe["finish_reason"] == "length"),
        "first_token_recorded": (
            streaming_probe["first_token_monotonic_s"] is not None
        ),
        "timing_order_is_valid": (
            streaming_probe["started_monotonic_s"]
            <= streaming_probe["first_token_monotonic_s"]
            <= streaming_probe["completed_monotonic_s"]
        ),
        "ttft_is_positive": (streaming_probe["ttft_s"] > 0),
        "latency_exceeds_ttft": (
            streaming_probe["latency_s"] >= streaming_probe["ttft_s"]
        ),
        "output_is_nonempty": bool(streaming_probe["output_text"]),
    },
    name="passed",
)

display(streaming_probe_invariants.to_frame())

if not bool(streaming_probe_invariants.all()):
    raise AssertionError("The streaming completion probe failed one or more invariants")

transport_comparison = pd.Series(
    {
        "nonstreaming_output_characters": len(completion_probe["output_text"]),
        "streaming_output_characters": len(streaming_probe["output_text"]),
        "outputs_match": (
            streaming_probe["output_text"] == completion_probe["output_text"]
        ),
    },
    name="value",
)

display(transport_comparison.to_frame())

print("Streaming completion contract verified.")

,passed
status_is_ok,True
http_status_is_200,True
model_matches,True
prompt_tokens_match,True
completion_tokens_match,True
finish_reason_is_length,True
first_token_recorded,True
timing_order_is_valid,True
ttft_is_positive,True
latency_exceeds_ttft,True


,value
nonstreaming_output_characters,92
streaming_output_characters,92
outputs_match,True


Streaming completion contract verified.


In [61]:
def stop_notebook_server(
    *,
    container_name: str,
    process: subprocess.Popen[str] | None,
    log_file,
    timeout_s: float = 30.0,
) -> dict[str, object]:
    cleanup_started_s = time.monotonic()

    stop_result = subprocess.run(
        [
            "docker",
            "stop",
            "--time",
            str(int(timeout_s)),
            container_name,
        ],
        text=True,
        capture_output=True,
        timeout=timeout_s + 10.0,
        check=False,
    )

    process_returncode = None

    if process is not None:
        try:
            process_returncode = process.wait(timeout=10.0)
        except subprocess.TimeoutExpired:
            process.terminate()
            try:
                process_returncode = process.wait(timeout=5.0)
            except subprocess.TimeoutExpired:
                process.kill()
                process_returncode = process.wait(timeout=5.0)

    if log_file is not None and not log_file.closed:
        log_file.flush()
        log_file.close()

    return {
        "container_name": container_name,
        "docker_stop_returncode": stop_result.returncode,
        "docker_stop_stdout": stop_result.stdout.strip(),
        "docker_stop_stderr": stop_result.stderr.strip(),
        "process_returncode": process_returncode,
        "cleanup_duration_s": (time.monotonic() - cleanup_started_s),
    }


print(
    "Cleanup helper defined. Do not run it until measurements "
    "are complete or the server must be stopped."
)

Cleanup helper defined. Do not run it until measurements are complete or the server must be stopped.


### Warm-up and raw result schema

In [65]:
WARMUP_PROMPT_ID = f"prompt_{max(PROMPT_TOKEN_COUNTS)}"
WARMUP_OUTPUT_TOKENS = max(GENERATED_TOKEN_COUNTS)

WARMUP_RESULTS: list[dict[str, object]] = []

for warmup_index in range(1, WARMUP_COUNT + 1):
    result = send_streaming_completion_request(
        endpoint=ENDPOINT,
        model=MODEL,
        prompt_id=WARMUP_PROMPT_ID,
        prompt=PROMPTS[WARMUP_PROMPT_ID],
        requested_output_tokens=WARMUP_OUTPUT_TOKENS,
        timeout_s=REQUEST_TIMEOUT_S,
        sampling=SAMPLING,
    )

    warmup_result = {
        "phase": "warmup",
        "warmup_index": warmup_index,
        **result,
    }
    WARMUP_RESULTS.append(warmup_result)

    print(
        f"Warm-up {warmup_index}/{WARMUP_COUNT}: "
        f"status={result['status']}, "
        f"ttft={result['ttft_s']!r}, "
        f"latency={result['latency_s']!r}"
    )

warmup_results = pd.DataFrame(WARMUP_RESULTS)

warmup_results[
    [
        "warmup_index",
        "prompt_tokens",
        "returned_output_tokens",
        "finish_reason",
        "ttft_s",
        "generation_interval_s",
        "latency_s",
        "output_tokens_per_s",
        "status",
        "error",
    ]
]

Warm-up 1/3: status=ok, ttft=0.0876850439235568, latency=6.529098836006597
Warm-up 2/3: status=ok, ttft=0.056471911957487464, latency=6.502519309986383
Warm-up 3/3: status=ok, ttft=0.055686293868348, latency=6.507996530039236


,warmup_index,prompt_tokens,returned_output_tokens,finish_reason,ttft_s,generation_interval_s,latency_s,output_tokens_per_s,status,error
0,1,512,128,length,0.087685,6.441414,6.529099,19.871414,ok,None
1,2,512,128,length,0.056472,6.446047,6.502519,19.857130,ok,None
2,3,512,128,length,0.055686,6.452310,6.507997,19.837856,ok,None


In [ ]:
expected_warmup_prompt_tokens = len(PROMPT_TOKEN_IDS[WARMUP_PROMPT_ID])

warmup_invariants = pd.Series(
    {
        "warmup_count_matches": (len(warmup_results) == WARMUP_COUNT),
        "all_statuses_are_ok": bool(warmup_results["status"].eq("ok").all()),
        "all_http_statuses_are_200": bool(warmup_results["status_code"].eq(200).all()),
        "all_models_match": bool(warmup_results["response_model"].eq(MODEL).all()),
        "all_prompt_counts_match": bool(
            warmup_results["prompt_tokens"].eq(expected_warmup_prompt_tokens).all()
        ),
        "all_output_counts_match": bool(
            warmup_results["returned_output_tokens"].eq(WARMUP_OUTPUT_TOKENS).all()
        ),
        "all_finish_reasons_are_length": bool(
            warmup_results["finish_reason"].eq("length").all()
        ),
        "all_ttft_values_are_positive": bool(warmup_results["ttft_s"].gt(0).all()),
        "all_latencies_exceed_ttft": bool(
            warmup_results["latency_s"].ge(warmup_results["ttft_s"]).all()
        ),
        "all_outputs_are_nonempty": bool(warmup_results["output_text"].map(bool).all()),
    },
    name="passed",
)

display(warmup_invariants.to_frame())

if not bool(warmup_invariants.all()):
    raise AssertionError("The warm-up phase failed one or more invariants")

,passed
warmup_count_matches,True
all_statuses_are_ok,True
all_http_statuses_are_200,True
all_models_match,True
all_prompt_counts_match,True
all_output_counts_match,True
all_finish_reasons_are_length,True
all_ttft_values_are_positive,True
all_latencies_exceed_ttft,True
all_outputs_are_nonempty,True


In [ ]:
warmup_metric_summary = (
    warmup_results[
        [
            "ttft_s",
            "generation_interval_s",
            "latency_s",
            "output_tokens_per_s",
        ]
    ]
    .agg(["min", "median", "max"])
    .T
)

warmup_determinism = pd.Series(
    {
        "unique_output_count": (warmup_results["output_text"].nunique(dropna=False)),
        "outputs_match_each_other": (
            warmup_results["output_text"].nunique(dropna=False) == 1
        ),
        "matches_streaming_probe": bool(
            warmup_results["output_text"].eq(streaming_probe["output_text"]).all()
        ),
    },
    name="value",
)

display(warmup_metric_summary)
display(warmup_determinism.to_frame())

print("Warm-up requests retained separately; none are benchmark measurements.")

,min,median,max
ttft_s,0.055686,0.056472,0.087685
generation_interval_s,6.441414,6.446047,6.452310
latency_s,6.502519,6.507997,6.529099
output_tokens_per_s,19.837856,19.857130,19.871414


,value
unique_output_count,1
outputs_match_each_other,True
matches_streaming_probe,False


Warm-up requests retained separately; none are benchmark measurements.


### Measured factorial sweep

Execute five repetitions of each prompt/output configuration in a deterministic randomized order.

Memory samples record device-wide `nvidia-smi` memory immediately before and after each request. They are not request-peak measurements and may include memory belonging to other GPU users.

For this notebook, `output_correct` means that the request satisfied its model, token-count, finish-reason, and within-configuration deterministic-consistency checks. It does not evaluate semantic answer quality.

In [ ]:
from random import Random


benchmark_schedule_records = [
    {
        "trial": trial,
        "prompt_tokens_target": int(workload.prompt_tokens_target),
        "requested_output_tokens": int(workload.requested_output_tokens),
    }
    for trial in range(1, REPETITIONS + 1)
    for workload in workload_matrix.itertuples(index=False)
]

schedule_random = Random(SAMPLING["seed"])
schedule_random.shuffle(benchmark_schedule_records)

benchmark_schedule = pd.DataFrame(
    [
        {
            "execution_order": execution_order,
            "configuration": (
                f"p{record['prompt_tokens_target']}"
                f"_o{record['requested_output_tokens']}"
            ),
            **record,
        }
        for execution_order, record in enumerate(
            benchmark_schedule_records,
            start=1,
        )
    ]
)

if len(benchmark_schedule) != expected_measured_requests:
    raise AssertionError("Benchmark schedule has an unexpected size")


def read_device_memory_used() -> dict[str, object]:
    completed = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=memory.used",
            "--format=csv,noheader,nounits",
        ],
        text=True,
        capture_output=True,
        timeout=10.0,
        check=False,
    )

    if completed.returncode != 0:
        return {
            "memory_used_bytes": None,
            "memory_sample_error": (
                f"nvidia-smi exited with "
                f"{completed.returncode}: "
                f"{completed.stderr.strip()}"
            ),
        }

    values = [line.strip() for line in completed.stdout.splitlines() if line.strip()]

    if len(values) != 1:
        return {
            "memory_used_bytes": None,
            "memory_sample_error": (
                f"Expected one GPU memory value, received {values!r}"
            ),
        }

    try:
        used_mib = float(values[0])
    except ValueError:
        return {
            "memory_used_bytes": None,
            "memory_sample_error": (f"Could not parse GPU memory value {values[0]!r}"),
        }

    return {
        "memory_used_bytes": int(round(used_mib * 1024**2)),
        "memory_sample_error": None,
    }


benchmark_schedule

,execution_order,configuration,trial,prompt_tokens_target,requested_output_tokens
0,1,p32_o128,2,32,128
1,2,p32_o128,1,32,128
2,3,p32_o128,5,32,128
3,4,p32_o16,4,32,16
4,5,p512_o16,1,512,16
5,6,p512_o16,5,512,16
6,7,p512_o128,1,512,128
7,8,p512_o16,2,512,16
8,9,p32_o16,2,32,16
9,10,p512_o128,2,512,128


In [ ]:
MEASURED_RESULTS: list[dict[str, object]] = []

for row in benchmark_schedule.itertuples(index=False):
    prompt_id = f"prompt_{row.prompt_tokens_target}"

    memory_before = read_device_memory_used()

    request_result = send_streaming_completion_request(
        endpoint=ENDPOINT,
        model=MODEL,
        prompt_id=prompt_id,
        prompt=PROMPTS[prompt_id],
        requested_output_tokens=int(row.requested_output_tokens),
        timeout_s=REQUEST_TIMEOUT_S,
        sampling=SAMPLING,
    )

    memory_after = read_device_memory_used()

    request_valid = (
        request_result["status"] == "ok"
        and request_result["status_code"] == 200
        and request_result["response_model"] == MODEL
        and request_result["prompt_tokens"] == row.prompt_tokens_target
        and request_result["returned_output_tokens"] == row.requested_output_tokens
        and request_result["finish_reason"] == "length"
    )

    measured_result = {
        "phase": "measured",
        "execution_order": int(row.execution_order),
        "trial": int(row.trial),
        "configuration": row.configuration,
        "prompt_tokens_target": int(row.prompt_tokens_target),
        "gpu_memory_used_before_bytes": (memory_before["memory_used_bytes"]),
        "gpu_memory_used_after_bytes": (memory_after["memory_used_bytes"]),
        "gpu_memory_before_error": (memory_before["memory_sample_error"]),
        "gpu_memory_after_error": (memory_after["memory_sample_error"]),
        "request_valid": request_valid,
        **request_result,
    }

    MEASURED_RESULTS.append(measured_result)

    print(
        f"{row.execution_order:02d}/"
        f"{expected_measured_requests}: "
        f"{row.configuration}, "
        f"trial={row.trial}, "
        f"status={request_result['status']}, "
        f"ttft={request_result['ttft_s']!r}, "
        f"latency={request_result['latency_s']!r}"
    )

raw_results = (
    pd.DataFrame(MEASURED_RESULTS).sort_values("execution_order").reset_index(drop=True)
)

raw_results[
    [
        "execution_order",
        "trial",
        "configuration",
        "prompt_tokens",
        "requested_output_tokens",
        "returned_output_tokens",
        "ttft_s",
        "generation_interval_s",
        "latency_s",
        "output_tokens_per_s",
        "gpu_memory_used_before_bytes",
        "gpu_memory_used_after_bytes",
        "request_valid",
        "status",
        "error",
    ]
]

01/20: p32_o128, trial=2, status=ok, ttft=0.05526614794507623, latency=6.470993435941637
02/20: p32_o128, trial=1, status=ok, ttft=0.05375775997526944, latency=6.464640737045556
03/20: p32_o128, trial=5, status=ok, ttft=0.055670116096735, latency=6.470876306062564
04/20: p32_o16, trial=4, status=ok, ttft=0.0565220071002841, latency=0.8136346039827913
05/20: p512_o16, trial=1, status=ok, ttft=0.05551691586151719, latency=0.8187220108229667
06/20: p512_o16, trial=5, status=ok, ttft=0.05993817583657801, latency=0.8207127689383924
07/20: p512_o128, trial=1, status=ok, ttft=0.05802716314792633, latency=6.512755159055814
08/20: p512_o16, trial=2, status=ok, ttft=0.058305514976382256, latency=0.821519378805533
09/20: p32_o16, trial=2, status=ok, ttft=0.05643851892091334, latency=0.8123181380797178
10/20: p512_o128, trial=2, status=ok, ttft=0.059313485864549875, latency=6.547671335982159
11/20: p512_o128, trial=4, status=ok, ttft=0.056759863160550594, latency=6.5213719410821795
12/20: p512_o16

,execution_order,trial,configuration,prompt_tokens,requested_output_tokens,returned_output_tokens,ttft_s,generation_interval_s,latency_s,output_tokens_per_s,gpu_memory_used_before_bytes,gpu_memory_used_after_bytes,request_valid,status,error
0,1,2,p32_o128,32,128,128,0.055266,6.415727,6.470993,19.950973,None,None,True,ok,None
1,2,1,p32_o128,32,128,128,0.053758,6.410883,6.464641,19.966048,None,None,True,ok,None
2,3,5,p32_o128,32,128,128,0.055670,6.415206,6.470876,19.952593,None,None,True,ok,None
3,4,4,p32_o16,32,16,16,0.056522,0.757113,0.813635,21.132920,None,None,True,ok,None
4,5,1,p512_o16,512,16,16,0.055517,0.763205,0.818722,20.964221,None,None,True,ok,None
5,6,5,p512_o16,512,16,16,0.059938,0.760775,0.820713,21.031197,None,None,True,ok,None
6,7,1,p512_o128,512,128,128,0.058027,6.454728,6.512755,19.830425,None,None,True,ok,None
7,8,2,p512_o16,512,16,16,0.058306,0.763214,0.821519,20.963980,None,None,True,ok,None
8,9,2,p32_o16,32,16,16,0.056439,0.755880,0.812318,21.167392,None,None,True,ok,None
9,10,2,p512_o128,512,128,128,0.059313,6.488358,6.547671,19.727642,None,None,True,ok,None


In [ ]:
raw_results["end_to_end_output_tokens_per_s"] = (
    raw_results["returned_output_tokens"] / raw_results["latency_s"]
)

raw_results["output_consistent_with_configuration"] = raw_results.groupby(
    "configuration"
)["output_text"].transform(lambda values: values.eq(values.iloc[0]))

raw_results["output_correct"] = (
    raw_results["request_valid"] & raw_results["output_consistent_with_configuration"]
)

configuration_counts = raw_results.groupby("configuration").size().sort_index()

raw_result_invariants = pd.Series(
    {
        "measured_request_count_matches": (
            len(raw_results) == expected_measured_requests
        ),
        "execution_order_is_unique": (raw_results["execution_order"].is_unique),
        "client_request_ids_are_unique": (raw_results["client_request_id"].is_unique),
        "each_configuration_has_five_trials": bool(
            configuration_counts.eq(REPETITIONS).all()
        ),
        "all_requests_are_valid": bool(raw_results["request_valid"].all()),
        "all_outputs_are_consistent": bool(
            raw_results["output_consistent_with_configuration"].all()
        ),
        "all_outputs_are_correct": bool(raw_results["output_correct"].all()),
    },
    name="passed",
)

memory_sampling_summary = pd.Series(
    {
        "before_samples_present": int(
            raw_results["gpu_memory_used_before_bytes"].notna().sum()
        ),
        "after_samples_present": int(
            raw_results["gpu_memory_used_after_bytes"].notna().sum()
        ),
        "before_sample_errors": int(
            raw_results["gpu_memory_before_error"].notna().sum()
        ),
        "after_sample_errors": int(raw_results["gpu_memory_after_error"].notna().sum()),
    },
    name="value",
)

determinism_summary = raw_results.groupby(
    "configuration",
    as_index=False,
).agg(
    request_count=(
        "client_request_id",
        "size",
    ),
    unique_output_count=(
        "output_text",
        "nunique",
    ),
    all_outputs_correct=(
        "output_correct",
        "all",
    ),
)

display(configuration_counts.to_frame("count"))
display(raw_result_invariants.to_frame())
display(memory_sampling_summary.to_frame())
display(determinism_summary)

if not bool(raw_result_invariants.all()):
    raise AssertionError("The measured sweep failed one or more raw-result invariants")

print("Raw measured results verified. Aggregation has not yet been performed.")

,count
configuration,
p32_o128,5
p32_o16,5
p512_o128,5
p512_o16,5


,passed
measured_request_count_matches,True
execution_order_is_unique,True
client_request_ids_are_unique,True
each_configuration_has_five_trials,True
all_requests_are_valid,True
all_outputs_are_consistent,True
all_outputs_are_correct,True


,value
before_samples_present,0
after_samples_present,0
before_sample_errors,20
after_sample_errors,20


,configuration,request_count,unique_output_count,all_outputs_correct
0,p32_o128,5,1,True
1,p32_o16,5,1,True
2,p512_o128,5,1,True
3,p512_o16,5,1,True


Raw measured results verified. Aggregation has not yet been performed.


In [ ]:
memory_sampling_errors = pd.concat(
    [
        raw_results["gpu_memory_before_error"].rename("error"),
        raw_results["gpu_memory_after_error"].rename("error"),
    ],
    ignore_index=True,
).value_counts(dropna=False)

memory_sampling_errors.to_frame("count")

,count
error,
Could not parse GPU memory value '[N/A]',40


In [ ]:
import re


if server_log_file is not None and not server_log_file.closed:
    server_log_file.flush()

current_server_log = SERVER_LOG_PATH.read_text(
    encoding="utf-8",
    errors="replace",
)

memory_accounting_line = next(
    (
        line
        for line in current_server_log.splitlines()
        if "Free memory on device" in line and "Current kv cache memory in use" in line
    ),
    None,
)

if memory_accounting_line is None:
    raise RuntimeError("The server log does not contain vLLM memory accounting")

memory_pattern = re.compile(
    r"Free memory on device "
    r"\((?P<free_gib>[\d.]+)/"
    r"(?P<total_gib>[\d.]+) GiB\)"
    r".*?"
    r"Desired GPU memory utilization is "
    r"\((?P<utilization>[\d.]+), "
    r"(?P<budget_gib>[\d.]+) GiB\)"
    r".*?"
    r"Actual usage is "
    r"(?P<weights_gib>[\d.]+) GiB for weight, "
    r"(?P<activation_gib>[\d.]+) GiB "
    r"for peak activation, "
    r"(?P<non_torch_gib>[\d.]+) GiB "
    r"for non-torch memory, and "
    r"(?P<cudagraph_gib>[\d.]+) GiB "
    r"for CUDAGraph memory\."
    r".*?"
    r"Current kv cache memory in use is "
    r"(?P<kv_cache_gib>[\d.]+) GiB\."
)

memory_match = memory_pattern.search(memory_accounting_line)

if memory_match is None:
    print(memory_accounting_line)
    raise RuntimeError("Could not parse vLLM memory accounting")

server_memory_accounting = {
    key: float(value) for key, value in memory_match.groupdict().items()
}

server_memory_accounting["accounted_total_gib"] = sum(
    server_memory_accounting[key]
    for key in (
        "weights_gib",
        "activation_gib",
        "non_torch_gib",
        "cudagraph_gib",
        "kv_cache_gib",
    )
)

pd.Series(
    server_memory_accounting,
    name="value",
)

free_gib               110.00
total_gib              121.69
utilization              0.20
budget_gib              24.34
weights_gib             15.19
activation_gib           0.39
non_torch_gib            0.65
cudagraph_gib            1.07
kv_cache_gib             8.11
accounted_total_gib     25.41
Name: value, dtype: float64

In [ ]:
kv_cache_size_match = re.search(
    r"GPU KV cache size: "
    r"(?P<tokens>[\d,]+) tokens",
    current_server_log,
)

kv_concurrency_match = re.search(
    r"Maximum concurrency for "
    r"1,024 tokens per request: "
    r"(?P<concurrency>[\d.]+)x",
    current_server_log,
)

if kv_cache_size_match is None:
    raise RuntimeError("GPU KV-cache token capacity was not found")

if kv_concurrency_match is None:
    raise RuntimeError("KV-cache concurrency was not found")

server_cache_capacity = pd.Series(
    {
        "max_model_len": VLLM_MAX_MODEL_LEN,
        "kv_cache_tokens": int(kv_cache_size_match.group("tokens").replace(",", "")),
        "maximum_concurrency_at_max_model_len": (
            float(kv_concurrency_match.group("concurrency"))
        ),
        "configured_concurrency": 1,
    },
    name="value",
)

display(server_cache_capacity)

if server_memory_accounting["utilization"] != VLLM_GPU_MEMORY_UTILIZATION:
    raise AssertionError(
        "The log memory utilization does not match the notebook configuration"
    )

print(
    "Server startup memory accounting verified. "
    "Per-request nvidia-smi samples remain "
    "unavailable and are not interpreted."
)

max_model_len                             1024.0
kv_cache_tokens                         148582.0
maximum_concurrency_at_max_model_len       145.1
configured_concurrency                       1.0
Name: value, dtype: float64

Server startup memory accounting verified. Per-request nvidia-smi samples remain unavailable and are not interpreted.


### Aggregation schema

In [ ]:
AGGREGATE_METRICS = (
    "ttft_s",
    "generation_interval_s",
    "latency_s",
    "output_tokens_per_s",
    "end_to_end_output_tokens_per_s",
)

aggregate_records: list[dict[str, object]] = []

for (
    configuration,
    prompt_tokens,
    requested_output_tokens,
), group in raw_results.groupby(
    [
        "configuration",
        "prompt_tokens_target",
        "requested_output_tokens",
    ],
    sort=True,
):
    failure_count = int((~group["request_valid"]).sum())

    for metric in AGGREGATE_METRICS:
        values = group.loc[
            group["request_valid"],
            metric,
        ].dropna()

        aggregate_records.append(
            {
                "configuration": configuration,
                "prompt_tokens": int(prompt_tokens),
                "requested_output_tokens": int(requested_output_tokens),
                "metric": metric,
                "minimum": values.min(),
                "median": values.median(),
                "p90": values.quantile(0.90),
                "p99": values.quantile(0.99),
                "count": int(values.size),
                "failures": failure_count,
            }
        )

aggregates = (
    pd.DataFrame(aggregate_records)
    .sort_values(
        [
            "prompt_tokens",
            "requested_output_tokens",
            "metric",
        ]
    )
    .reset_index(drop=True)
)

aggregates

,configuration,prompt_tokens,requested_output_tokens,metric,minimum,median,p90,p99,count,failures
0,p32_o16,32,16,end_to_end_output_tokens_per_s,19.516546,19.624174,19.683969,19.695442,5,0
1,p32_o16,32,16,generation_interval_s,0.755880,0.759232,0.762067,0.762720,5,0
2,p32_o16,32,16,latency_s,0.812318,0.815321,0.818856,0.819721,5,0
3,p32_o16,32,16,output_tokens_per_s,20.975553,21.073931,21.153603,21.166013,5,0
4,p32_o16,32,16,ttft_s,0.056089,0.056439,0.056823,0.057004,5,0
5,p32_o128,32,128,end_to_end_output_tokens_per_s,19.670434,19.780579,19.792386,19.799254,5,0
6,p32_o128,32,128,generation_interval_s,6.410883,6.415727,6.447384,6.451282,5,0
7,p32_o128,32,128,latency_s,6.464641,6.470993,6.502853,6.506791,5,0
8,p32_o128,32,128,output_tokens_per_s,19.839685,19.950973,19.960666,19.965510,5,0
9,p32_o128,32,128,ttft_s,0.053758,0.055404,0.055607,0.055664,5,0


In [ ]:
expected_aggregate_rows = (
    len(PROMPT_TOKEN_COUNTS) * len(GENERATED_TOKEN_COUNTS) * len(AGGREGATE_METRICS)
)

aggregate_invariants = pd.Series(
    {
        "row_count_matches": (len(aggregates) == expected_aggregate_rows),
        "all_counts_match_repetitions": bool(aggregates["count"].eq(REPETITIONS).all()),
        "no_failures": bool(aggregates["failures"].eq(0).all()),
        "all_minima_are_positive": bool(aggregates["minimum"].gt(0).all()),
        "minimum_not_above_median": bool(
            aggregates["minimum"].le(aggregates["median"]).all()
        ),
        "median_not_above_p90": bool(aggregates["median"].le(aggregates["p90"]).all()),
        "p90_not_above_p99": bool(aggregates["p90"].le(aggregates["p99"]).all()),
    },
    name="passed",
)

display(aggregate_invariants.to_frame())

if not bool(aggregate_invariants.all()):
    raise AssertionError("Aggregate results failed one or more invariants")

print(
    "Aggregate results verified. With five "
    "observations per configuration, p90 and "
    "p99 are descriptive sample summaries only."
)

,passed
row_count_matches,True
all_counts_match_repetitions,True
no_failures,True
all_minima_are_positive,True
minimum_not_above_median,True
median_not_above_p90,True
p90_not_above_p99,True


Aggregate results verified. With five observations per configuration, p90 and p99 are descriptive sample summaries only.


In [76]:
median_results = (
    aggregates.pivot_table(
        index=[
            "prompt_tokens",
            "requested_output_tokens",
        ],
        columns="metric",
        values="median",
        aggfunc="first",
    )
    .reset_index()
    .sort_values(
        [
            "prompt_tokens",
            "requested_output_tokens",
        ]
    )
)

median_results

metric,prompt_tokens,requested_output_tokens,end_to_end_output_tokens_per_s,generation_interval_s,latency_s,output_tokens_per_s,ttft_s
0,32,16,19.624174,0.759232,0.815321,21.073931,0.056439
1,32,128,19.780579,6.415727,6.470993,19.950973,0.055404
2,512,16,19.476108,0.763205,0.821519,20.964221,0.059594
3,512,128,19.572365,6.480637,6.539833,19.751146,0.058874


In [ ]:
median_lookup = median_results.set_index(
    [
        "prompt_tokens",
        "requested_output_tokens",
    ]
)

prompt_length_effects = []

short_prompt = min(PROMPT_TOKEN_COUNTS)
long_prompt = max(PROMPT_TOKEN_COUNTS)

for output_tokens in GENERATED_TOKEN_COUNTS:
    short_ttft = median_lookup.loc[
        (short_prompt, output_tokens),
        "ttft_s",
    ]
    long_ttft = median_lookup.loc[
        (long_prompt, output_tokens),
        "ttft_s",
    ]

    prompt_length_effects.append(
        {
            "requested_output_tokens": (output_tokens),
            "short_prompt_tokens": short_prompt,
            "long_prompt_tokens": long_prompt,
            "short_prompt_median_ttft_s": (short_ttft),
            "long_prompt_median_ttft_s": (long_ttft),
            "ttft_difference_s": (long_ttft - short_ttft),
            "ttft_ratio": (long_ttft / short_ttft),
        }
    )

prompt_length_effects = pd.DataFrame(prompt_length_effects)

output_length_effects = []

short_output = min(GENERATED_TOKEN_COUNTS)
long_output = max(GENERATED_TOKEN_COUNTS)

for prompt_tokens in PROMPT_TOKEN_COUNTS:
    short_row = median_lookup.loc[(prompt_tokens, short_output)]
    long_row = median_lookup.loc[(prompt_tokens, long_output)]

    output_length_effects.append(
        {
            "prompt_tokens": prompt_tokens,
            "short_output_tokens": short_output,
            "long_output_tokens": long_output,
            "short_output_median_latency_s": (short_row["latency_s"]),
            "long_output_median_latency_s": (long_row["latency_s"]),
            "latency_difference_s": (long_row["latency_s"] - short_row["latency_s"]),
            "latency_ratio": (long_row["latency_s"] / short_row["latency_s"]),
            "short_output_generation_tokens_per_s": (short_row["output_tokens_per_s"]),
            "long_output_generation_tokens_per_s": (long_row["output_tokens_per_s"]),
            "short_output_end_to_end_tokens_per_s": (
                short_row["end_to_end_output_tokens_per_s"]
            ),
            "long_output_end_to_end_tokens_per_s": (
                long_row["end_to_end_output_tokens_per_s"]
            ),
        }
    )

output_length_effects = pd.DataFrame(output_length_effects)

display(prompt_length_effects)
display(output_length_effects)

,requested_output_tokens,short_prompt_tokens,long_prompt_tokens,short_prompt_median_ttft_s,long_prompt_median_ttft_s,ttft_difference_s,ttft_ratio
0,16,32,512,0.056439,0.059594,0.003156,1.055913
1,128,32,512,0.055404,0.058874,0.003470,1.062634


,prompt_tokens,short_output_tokens,long_output_tokens,short_output_median_latency_s,long_output_median_latency_s,latency_difference_s,latency_ratio,short_output_generation_tokens_per_s,long_output_generation_tokens_per_s,short_output_end_to_end_tokens_per_s,long_output_end_to_end_tokens_per_s
0,32,16,128,0.815321,6.470993,5.655672,7.936744,21.073931,19.950973,19.624174,19.780579
1,512,16,128,0.821519,6.539833,5.718314,7.960656,20.964221,19.751146,19.476108,19.572365


In [ ]:
def rank_correlation(
    left: pd.Series,
    right: pd.Series,
) -> float:
    return left.rank().corr(right.rank())


execution_order_diagnostics = pd.DataFrame(
    [
        {
            "configuration": configuration,
            "request_count": len(group),
            "first_execution_order": int(group["execution_order"].min()),
            "last_execution_order": int(group["execution_order"].max()),
            "ttft_order_rank_correlation": (
                rank_correlation(
                    group["execution_order"],
                    group["ttft_s"],
                )
            ),
            "latency_order_rank_correlation": (
                rank_correlation(
                    group["execution_order"],
                    group["latency_s"],
                )
            ),
        }
        for configuration, group in raw_results.groupby(
            "configuration",
            sort=True,
        )
    ]
)

execution_order_diagnostics

,configuration,request_count,first_execution_order,last_execution_order,ttft_order_rank_correlation,latency_order_rank_correlation
0,p32_o128,5,1,18,0.6,0.7
1,p32_o16,5,4,19,0.1,0.9
2,p512_o128,5,7,20,0.1,0.4
3,p512_o16,5,5,13,0.7,1.0


### Cleanup guidance

Stop only the `server_process` handle created by this notebook, first requesting graceful termination and then applying a bounded wait. Never use unscoped process-kill commands.

In [79]:
cleanup_result = stop_notebook_server(
    container_name=SERVER_CONTAINER_NAME,
    process=server_process,
    log_file=server_log_file,
)

pd.Series(cleanup_result, name="value")

container_name                               dgx-spark-single-node-68ca5e09
docker_stop_returncode                                                    0
docker_stop_stdout        Flag --time has been deprecated, use --timeout...
docker_stop_stderr                                                         
process_returncode                                                        0
cleanup_duration_s                                                 1.614308
Name: value, dtype: object

## Observations

### Server startup and memory

The single-node vLLM server became ready in 245.133 seconds and advertised the configured `google/gemma-4-E4B-it` model.

The server used:

- `gpu_memory_utilization=0.20`;
- a maximum model length of 1,024 tokens;
- tensor-parallel size 1;
- the Triton attention backend selected by vLLM for Gemma 4's heterogeneous attention-head dimensions.

vLLM reported a 24.34 GiB memory-sizing budget. Its startup log separately reported:

- 15.19 GiB for model weights;
- 0.39 GiB for peak activation memory;
- 0.65 GiB for non-PyTorch memory;
- 1.07 GiB for CUDA graphs;
- 8.11 GiB for the KV cache.

The KV cache held 148,582 tokens and vLLM reported a maximum concurrency of 145.1 sequences at the configured 1,024-token maximum length. The experiment itself used concurrency 1.

The reported component values sum to 25.41 GiB, but the notebook does not establish that these categories are mutually exclusive or measured at the same allocation boundary. Their sum is therefore not interpreted as physical memory consumption or as evidence that the sizing budget was exceeded.

`nvidia-smi` returned `[N/A]` for every attempted device-memory sample. Per-request memory use was consequently not measured. The startup values above are vLLM's own allocation and profiling records.

### API and warm-up validation

Both non-streaming and streaming completion probes succeeded. The probes verified:

- HTTP status 200;
- served-model identity;
- exact prompt and output token accounting;
- length-based completion;
- non-empty output;
- valid client-observed timing boundaries.

The streaming probe measured:

- TTFT: 0.08134 seconds;
- generation interval: 0.75611 seconds;
- end-to-end latency: 0.83745 seconds;
- generation throughput: 21.161 output tokens per second.

The streaming and non-streaming probes returned identical text.

Three retained 512-prompt-token, 128-output-token warm-ups all succeeded and returned identical output. Their TTFT values were 0.08769, 0.05647, and 0.05569 seconds. Their end-to-end latencies ranged from 6.50252 to 6.52910 seconds, while generation throughput ranged from 19.8379 to 19.8714 output tokens per second.

### Measured factorial sweep

All 20 measured requests succeeded. Each of the four configurations had five repetitions, and every request returned the intended prompt and output token counts with finish reason `length`.

Median results were:

| Prompt tokens | Output tokens | TTFT (s) | Generation interval (s) | Latency (s) | Generation tokens/s | End-to-end tokens/s |
|---:|---:|---:|---:|---:|---:|---:|
| 32 | 16 | 0.05644 | 0.75923 | 0.81532 | 21.0739 | 19.6242 |
| 32 | 128 | 0.05540 | 6.41573 | 6.47099 | 19.9510 | 19.7806 |
| 512 | 16 | 0.05959 | 0.76321 | 0.82152 | 20.9642 | 19.4761 |
| 512 | 128 | 0.05887 | 6.48064 | 6.53983 | 19.7511 | 19.5724 |

Increasing prompt length from 32 to 512 tokens increased median TTFT by:

- 0.00316 seconds for 16-token output;
- 0.00347 seconds for 128-token output.

Increasing requested output length from 16 to 128 tokens increased median end-to-end latency by:

- 5.65567 seconds, or 7.9367×, for 32-token prompts;
- 5.71831 seconds, or 7.9607×, for 512-token prompts.

Within every configuration, all five outputs were identical. No request failed, timed out, returned an incorrect token count, or violated the timing invariants.

The per-configuration execution-order diagnostics showed positive rank correlations for several metrics, including latency correlations of 0.9 for `p32_o16` and 1.0 for `p512_o16`. With only five observations per configuration, these are descriptive indications of possible time-order drift rather than evidence of a specific thermal, scheduling, or runtime cause.

The p90 and p99 values are retained as descriptive summaries of five observations per configuration. They are not treated as stable tail-latency estimates.

## Explanation

The prediction was partly confirmed.

Longer prompts produced consistently higher median TTFT. Moving from 32 to 512 prompt tokens increased TTFT by approximately 3.2–3.5 milliseconds, or 5.6–6.3 percent. This is directionally consistent with additional prefill work.

The effect of prompt length was not isolated to TTFT. For the 128-token generation, the longer prompt also increased the median generation interval and reduced generation throughput. Autoregressive decoding attends over the existing context, so a longer prompt can affect work performed during later decode steps as well as initial prefill. The notebook does not contain kernel-level measurements that would attribute the difference to a particular operation.

Generated length had the dominant effect on total latency. Increasing output length by 8× increased end-to-end latency by approximately 7.94–7.96×. The ratio is slightly below 8 because request setup, prefill, and protocol overhead do not scale with the number of generated tokens.

The throughput prediction depends on the metric definition:

- Generation-interval throughput was higher for 16-token generations than for 128-token generations. This contradicts the prediction that short generations would have lower output-token throughput.
- End-to-end throughput was slightly higher for 128-token generations. This is consistent with fixed TTFT and request overhead being amortized across more output tokens.

The distinction is important: excluding TTFT exposes decode behavior, while including TTFT measures the client-observed rate for the complete request.

The first formal warm-up had a higher TTFT than the next two warm-ups under the same 512-to-128 workload. This supports the presence of residual one-time or early-request work. It does not directly prove the original claim about the first request after readiness, because non-streaming and streaming diagnostic probes had already run.

Output determinism was confirmed within the measured scope. Five requests for every configuration returned identical text under greedy sampling with the same seed, and the equivalent streaming and non-streaming probes also matched. This does not establish determinism across server restarts, software versions, hardware platforms, or distributed execution.

The randomized schedule reduced direct confounding between configuration and execution time, but the positive order correlations show that small temporal drift may remain. Repeated full benchmark runs would be required to determine whether this is reproducible.

No conclusion is drawn about compute saturation, memory-bandwidth saturation, or kernel efficiency. The notebook measures client-observed timing and vLLM startup accounting, not GPU utilization, memory traffic, or kernel activity.

## Connection to LLMs

This experiment establishes a controlled single-node inference denominator for the pinned Gemma 4 model.

The measurements separate two important phases of autoregressive inference:

1. prompt processing contributes to client-observed TTFT;
2. token-by-token decoding dominates latency for longer generated outputs.

The results also show why throughput must be defined explicitly. Generation-only token rate and end-to-end token rate answered slightly different questions for the same requests.

The 0.20 vLLM memory setting reserved far more KV-cache capacity than the concurrency-1 workload required. This confirms that vLLM's memory configuration represents a serving-capacity envelope rather than memory proportional to the single active request.

Later distributed comparisons should preserve:

- the pinned model revision;
- the 1,024-token maximum model length;
- prompt serialization and token counts;
- sampling configuration;
- warm-up boundary;
- streaming timing definitions;
- randomized request schedule;
- raw failure retention.

Differences observed after introducing tensor parallelism can then be compared against this baseline without silently changing the client workload or metric definitions.

## Further Exploration

The next controlled experiments are:

1. Repeat the complete single-node sweep across multiple fresh server launches to quantify run-to-run startup, TTFT, latency, and execution-order variation.
2. Add unified-memory telemetry that is supported on DGX Spark, such as bounded `/proc/meminfo`, cgroup, process-RSS, or another validated GB10-compatible source. Keep endpoint samples distinct from peak memory.
3. Measure GPU activity and kernel timing separately before making compute-bound or memory-bandwidth-bound claims.
4. Introduce concurrency and batching as separate experiments rather than changing the concurrency-1 baseline.
5. Launch the pinned model with two-node tensor parallelism and verify model identity, worker placement, readiness, output correctness, and teardown before measuring performance.
6. Compare single-node and distributed results using the same four prompt/output configurations and the same metric definitions.
7. Investigate whether the small execution-order correlations recur across independent runs or disappear as sampling variation.